# Listar usuarios por HTTP nativo

## Preparación y recorrido

Usar solo la copia de ensayo con datos ficticios. Mantener el servidor SQL del ensayo y el cliente HTTP en `http://127.0.0.1:3004` abiertos. Kernel **Python 3 (ipykernel)**. Ejecutar secuencialmente crear, listar, consultar, modificar y borrar; repetir listar para comprobar el vaciado. No se presupone ID 1.

Esta celda ejecuta Node mediante Python y llama a la API por HTTP; no importa la lógica ni consulta SQL directamente. Pulsar **Shift+Enter** para mostrar el estado y el cuerpo.

## Resultado esperado

200 y la lista; contiene Ada después de crear y queda vacía después de borrar.

## Anotaciones y errores

Ejecutar después de crear y repetir después de borrar para observar ambos resultados. Con SQL detenido: 500 `Could not list the users`.

Para comprobar SQL detenido, parar únicamente su proceso de ensayo y ejecutar de nuevo esta celda, manteniendo el HTTP abierto. En las operaciones por ID, si el listado ya devuelve error se envía `999999` para llegar igualmente a la operación y mostrar su mensaje específico. Ejecutar estas comprobaciones offline después del recorrido online.


In [1]:
import json
import subprocess

code = r"""
const origin = "http://127.0.0.1:3004";
const response = await fetch(origin + "/users", { method: "GET" });
const body = await response.text();
console.log(JSON.stringify({ status: response.status, body }));
"""
completed = subprocess.run(
    ["node", "--input-type=module", "--eval", code],
    capture_output=True,
    encoding="utf-8",
    timeout=30,
)
if completed.returncode != 0:
    raise RuntimeError(completed.stderr.strip())
result = json.loads(completed.stdout)
print(f"HTTP {result['status']}")
body = result["body"]
if not body:
    print("(sin contenido)")
else:
    value = json.loads(body)
    rows = value if isinstance(value, list) else [value]
    headers = list(rows[0]) if rows else []
    if not rows:
        print("Lista vacía: []")
    else:
        rows = [[str(row.get(header, "")).replace("\n", " ").replace("\r", " ").replace("\x1b", "") for header in headers] for row in rows]
        widths = [max(len(header), *(len(row[index]) for row in rows)) for index, header in enumerate(headers)]
        print("+" + "+".join("-" * (width + 2) for width in widths) + "+")
        print("| " + " | ".join("\x1b[1;36m" + header.ljust(width) + "\x1b[0m" for header, width in zip(headers, widths)) + " |")
        print("+" + "+".join("-" * (width + 2) for width in widths) + "+")
        for row in rows:
            print("| " + " | ".join(value.ljust(width) for value, width in zip(row, widths)) + " |")
        print("+" + "+".join("-" * (width + 2) for width in widths) + "+")


HTTP 200
+----+------+-----------------+---------------------+---------------------+
| id | name | email           | createdAt           | updatedAt           |
+----+------+-----------------+---------------------+---------------------+
| 1  | Ada  | ada@example.com | 2026-11-05 08:23:34 | 2026-11-05 08:23:34 |
+----+------+-----------------+---------------------+---------------------+
